In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('../../../src/zero_to_hero/part_02_makemore/names.txt', 'r').read().splitlines()

In [3]:
len(words)

32033

In [4]:
# We had S/E special tokens, but now just "."
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# ------------ now made respectable :) ---------------

In [6]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words, block_size):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1], block_size)
Xdev, Ydev = build_dataset(words[n1:n2], block_size)
Xte, Yte = build_dataset(words[n2:], block_size)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [40]:
# What if things are truly random???
# Note equality! If we have a uniform model, probability of anything right is 1/27, so you take logs and nll = log(27)
import numpy as np
logits_equal = torch.ones(Ytr.shape[0], 27)
loss = F.cross_entropy(logits_equal, Ytr)
print(loss, np.log(27))

tensor(3.2958) 3.295836866004329


In [75]:
# Now lets tune to have initial loss like this
# Skip a bunch of stuff to make the point
# Use he initialisation here
def compute_loss(params, Xtr, Ytr):
    mb_size = params["mb_size"]
    n_embedding = params["n_embedding"]
    #n_context = params["n_context"]
    n_context = 3
    n_hidden = params["n_hidden"]
    learning_rate = params["learning_rate"]
    weight_decay_lambda = params["reg"]
    steps = params["steps"]
    shrinker = 1/mb_size

    fan_in = [n_context * n_embedding, n_hidden]
    fan_out = [n_hidden, 27]
    shrinkers_glorot = [6 / np.sqrt(fan_in[0] + fan_out[0]), 6 / np.sqrt(fan_in[1] + fan_out[1])]
    shrinkers_he = [np.sqrt(2 / fan_in[0]), np.sqrt(2 / fan_in[1])]
    print(shrinkers_he)
    
    # Now define params
    g = torch.Generator().manual_seed(2147483647) # for reproducibility
    C = torch.randn((27, n_embedding), generator=g)  * shrinkers_he[0]
    W1 = torch.randn((n_context * n_embedding, n_hidden), generator=g) * shrinkers_he[0]
    b1 = torch.randn(n_hidden, generator=g) * shrinkers_he[0]
    W2 = torch.randn((n_hidden, 27), generator=g) * shrinkers_he[1]
    b2 = torch.randn(27, generator=g) * shrinkers_he[1]
    parameters = [C, W1, b1, W2, b2]
    
    for p in parameters:
        p.requires_grad = True
    
    for i in range(steps):
        #full batch
        ix = torch.randint(0, Xtr.shape[0], (mb_size,))
        # forward pass
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits,Ytr[ix]) + weight_decay_lambda * (W1**2).sum() + weight_decay_lambda * (W2**2).sum()
        print(loss.item())
        # Backward pass - should be none!
        for p in parameters:
            p.grad = None
        
        loss.backward()
        lr = learning_rate
        for p in parameters:
            p.data += -lr * p.grad

    emb = C[Xtr]
    h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) 
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr)

    tuned_params = [C, W1, b1, W2, b2]
    return loss, tuned_params

In [76]:
params = {'learning_rate': 0.5,
 'n_embedding': 3,
 'mb_size': 256,
 'n_hidden': 100,
 'reg': 0,
 'steps': 100}

In [77]:
_, tuned_params = compute_loss(params, Xtr, Ytr)

[np.float64(0.4714045207910317), np.float64(0.1414213562373095)]
3.7948830127716064
3.2788290977478027
3.1716578006744385
3.0505459308624268
2.913064479827881
2.944855213165283
2.9267024993896484
2.8052260875701904
2.8733177185058594
2.836635112762451
2.7991015911102295
2.7623250484466553
2.708944082260132
2.7129323482513428
2.719552755355835
2.7136411666870117
2.7516863346099854
2.808164358139038
2.7295196056365967
2.717308521270752
2.6449759006500244
2.7895963191986084
2.621901273727417
2.6374213695526123
2.603322744369507
2.6037349700927734
2.6499593257904053
2.6329739093780518
2.5603883266448975
2.634070873260498
2.622767925262451
2.6670331954956055
2.562457323074341
2.7400832176208496
2.6229889392852783
2.780451536178589
2.6052803993225098
2.7324273586273193
2.6158056259155273
2.729994773864746
2.5411715507507324
2.505504608154297
2.558755874633789
2.6985716819763184
2.5627973079681396
2.6414623260498047
2.654104471206665
2.6042895317077637
2.480879306793213
2.588801622390747
2.65

In [69]:
# Now lets tune to have initial loss like this
# Skip a bunch of stuff to make the point
# Use glorot initialisation here
def compute_loss(params, Xtr, Ytr):
    mb_size = params["mb_size"]
    n_embedding = params["n_embedding"]
    #n_context = params["n_context"]
    n_context = 3
    n_hidden = params["n_hidden"]
    learning_rate = params["learning_rate"]
    weight_decay_lambda = params["reg"]
    steps = params["steps"]
    shrinker = 1/mb_size

    fan_in = [n_context * n_embedding, n_hidden]
    fan_out = [n_hidden, 27]
    shrinkers_glorot = [6 / np.sqrt(fan_in[0] + fan_out[0]), 6 / np.sqrt(fan_in[1] + fan_out[1])]
    shrinkers_he = [np.sqrt(2 / fan_in[0]), np.sqrt(2 / fan_in[1])]
    print(shrinkers_he)
    
    # Now define params
    g = torch.Generator().manual_seed(2147483647) # for reproducibility
    C = torch.rand((27, n_embedding), generator=g)
    W1 = torch.rand((n_context * n_embedding, n_hidden), generator=g) * shrinkers_glorot[0] * 2 - shrinkers_glorot[0]
    b1 = torch.rand(n_hidden, generator=g)
    W2 = torch.rand((n_hidden, 27), generator=g) * shrinkers_glorot[1] * 2 - shrinkers_glorot[1]
    b2 = torch.rand(27, generator=g)
    parameters = [C, W1, b1, W2, b2]
    
    for p in parameters:
        p.requires_grad = True
    
    for i in range(steps):
        #full batch
        ix = torch.randint(0, Xtr.shape[0], (mb_size,))
        # forward pass
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits,Ytr[ix]) + weight_decay_lambda * (W1**2).sum() + weight_decay_lambda * (W2**2).sum()
        print(loss.item())
        # Backward pass - should be none!
        for p in parameters:
            p.grad = None
        
        loss.backward()
        lr = learning_rate
        for p in parameters:
            p.data += -lr * p.grad

    emb = C[Xtr]
    h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) 
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr)

    tuned_params = [C, W1, b1, W2, b2]
    return loss, tuned_params

In [70]:
params = {'learning_rate': 0.5,
 'n_embedding': 3,
 'mb_size': 256,
 'n_hidden': 100,
 'reg': 0,
 'steps': 100}

In [71]:
_, tuned_params = compute_loss(params, Xtr, Ytr)

[np.float64(0.4714045207910317), np.float64(0.1414213562373095)]
4.457167148590088
3.6930360794067383
4.718605041503906
4.236544132232666
3.134586811065674
3.039882183074951
3.1616952419281006
3.0657010078430176
3.2796733379364014
2.9443235397338867
2.8795557022094727
3.0043399333953857
2.7653653621673584
2.787980079650879
2.887295961380005
2.8234245777130127
2.875884532928467
2.6728837490081787
2.8391671180725098
2.681475877761841
2.8321588039398193
2.7810637950897217
2.813565254211426
2.78257155418396
2.9538533687591553
2.8971197605133057
2.7755541801452637
2.6380534172058105
2.7424440383911133
2.9570298194885254
3.0742173194885254
2.662877082824707
2.8138926029205322
2.814152956008911
2.626650094985962
2.769465684890747
2.7948334217071533
2.716549873352051
2.57977032661438
2.570467472076416
2.574185609817505
2.6502480506896973
2.693678379058838
2.6777865886688232
2.7449753284454346
2.784534215927124
2.6530086994171143
2.6061460971832275
2.632286787033081
2.6566972732543945
2.6781678

In [24]:
C, W1, b1, W2, b2 = tuned_params
n_embedding = params['n_embedding']
n_context = 3

tensor(3.2958) 3.295836866004329


In [26]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.2432, grad_fn=<NllLossBackward0>)

In [193]:
# Can tune hyperparameters (context, embedding)...
# Use better optimisation?
# He uses 100,000 0.1 and 0.01 - gets 2.16 on dev, 2.12 on train
# Hyperparameters to tune - (C-dimension, context size, number of neurons, proper learning rate?, batch size)
# Read paper and do things

In [192]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carlah.
qui.
hariffirri.
reviyah.
cassie.
mahnen.
amerric.
kaeli.
nellara.
chaiir.
kaleigph.
bron.
caten.
juzs.
lilah.
jadiquinathon.
jaryni.
jaxen.
ivuliwen.
emiia.
